# Building a GTFS flight feed for Victoria & NSW

This notebook uses the **`au_flights_gtfs`** package to build a valid GTFS feed of
scheduled flights between Victorian, NSW and ACT airports, using real timetable data
from the [AviationStack](https://aviationstack.com) `flightsFuture` endpoint.

**Date:** the feed targets **Wednesday, 10 June 2026**. Historical dates are blocked on
the available plan, and `flightsFuture` only serves dates ~a week ahead, so we use the
nearest queryable Wednesday and stamp the GTFS calendar with the same date.

**Pipeline:** configure → fetch (cached) → build GTFS tables → inspect → plot → export zip.

## 1. Setup

Install the package (from the repo root) and the notebook extras:

```bash
pip install -e ".[notebook]"
```

The cell below makes the package importable even if you have not installed it, by
adding the local `src/` directory to the path.

In [ ]:
import sys, pathlib

try:
    import au_flights_gtfs
except ModuleNotFoundError:
    # not installed: add the sibling src/ folder to the path
    src = pathlib.Path.cwd().parent / "src"
    sys.path.insert(0, str(src))
    import au_flights_gtfs

from au_flights_gtfs import Config, build_gtfs, AviationStackSource, GTFSBuilder, build_fallback_flights
print("au_flights_gtfs version", au_flights_gtfs.__version__)

## 2. Configure

The API key is read from the `AVIATIONSTACK_KEY` environment variable. If it is not set,
the cell prompts for it (without echoing) — **the key is never written to the notebook**.
Leave it blank to skip the API and use the built-in fallback timetable instead.

In [ ]:
import os, getpass

key = os.getenv("AVIATIONSTACK_KEY")
if not key:
    key = getpass.getpass("AviationStack API key (blank = use fallback timetable): ").strip()

cfg = Config(
    api_key=key,
    use_api=bool(key),
    fetch_date="2026-06-10",     # date queried from the API (a valid Wednesday)
    service_date="20260610",     # date stamped into the GTFS calendar
    service_id="SVC_20260610",
)
print("API enabled:", cfg.use_api)
print("Cache dir  :", cfg.cache_dir.resolve())
print("Output dir :", cfg.output_dir.resolve())

## 3. Fetch + build

`build_gtfs` runs the whole pipeline: it fetches every region airport's departures
(caching each response to disk so re-runs don't spend quota), dedupes codeshares,
chains multi-stop flights, builds the GTFS tables, and writes the folder + zip.

> **Note:** the free AviationStack plan is heavily rate-limited; a first, uncached run
> over all airports can take 20–40 minutes. Subsequent runs read from the cache and are
> almost instant.

In [ ]:
feed = build_gtfs(cfg, write=True)
print(feed.summary())

## 4. Inspect the GTFS tables with pandas

Every table is available as a list of dict-rows on `feed.tables`; we load a few into
DataFrames for a quick look.

In [ ]:
import pandas as pd

agency = pd.DataFrame(feed.tables["agency.txt"])
stops  = pd.DataFrame(feed.tables["stops.txt"])
routes = pd.DataFrame(feed.tables["routes.txt"])
trips  = pd.DataFrame(feed.tables["trips.txt"])
stop_times = pd.DataFrame(feed.tables["stop_times.txt"])

print("Airlines (agencies):")
display(agency)
print(f"\n{len(stops)} airports, {len(routes)} routes, {len(trips)} trips, {len(stop_times)} stop_times")

In [ ]:
# Busiest routes by number of trips
trips_per_route = (trips.groupby("route_id").size()
                        .rename("trips").reset_index()
                        .merge(routes[["route_id", "route_short_name", "route_long_name"]], on="route_id")
                        .sort_values("trips", ascending=False))
trips_per_route.head(15)

### A multi-stop flight

Flights with an intermediate stop are modelled as a single trip with 3+ `stop_times`,
just like a public-transport route that calls at several stops.

In [ ]:
counts = stop_times.groupby("trip_id").size().sort_values(ascending=False)
multi = counts[counts >= 3]
print(f"{len(multi)} multi-stop trips")
if len(multi):
    tid = multi.index[0]
    display(stop_times[stop_times["trip_id"] == tid]
            [["stop_sequence", "stop_id", "arrival_time", "departure_time", "pickup_type", "drop_off_type"]]
            .sort_values("stop_sequence"))

## 5. Plot the route network

Each shape is a straight line between airports. We draw the shapes and label the stops.

In [ ]:
import matplotlib.pyplot as plt

shapes = pd.DataFrame(feed.tables["shapes.txt"])
shapes[["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"]] = \
    shapes[["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"]].astype(float)
stops_f = stops.copy()
stops_f[["stop_lat", "stop_lon"]] = stops_f[["stop_lat", "stop_lon"]].astype(float)

fig, ax = plt.subplots(figsize=(9, 9))
for _, grp in shapes.groupby("shape_id"):
    grp = grp.sort_values("shape_pt_sequence")
    ax.plot(grp["shape_pt_lon"], grp["shape_pt_lat"], color="steelblue", alpha=0.4, lw=1)

ax.scatter(stops_f["stop_lon"], stops_f["stop_lat"], color="crimson", s=30, zorder=5)
for _, r in stops_f.iterrows():
    ax.annotate(r["stop_code"], (r["stop_lon"], r["stop_lat"]),
                textcoords="offset points", xytext=(4, 4), fontsize=8)

ax.set_title(f"VIC/NSW/ACT flight network — {cfg.service_date}")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout(); plt.show()

## 6. Output

The GTFS folder and zip have already been written by `build_gtfs(..., write=True)`.
Import `gtfs_flights.zip` into a validator such as the
[Canonical GTFS Validator](https://github.com/MobilityData/gtfs-validator) or a trip
planner like OpenTripPlanner.

In [ ]:
import zipfile
print("Folder:", cfg.output_dir.resolve())
print("Zip   :", cfg.zip_path.resolve())
with zipfile.ZipFile(cfg.zip_path) as zf:
    for info in zf.infolist():
        print(f"  {info.filename:<22} {info.file_size:>8} bytes")